# S30 - EA2 Unidad 1 - Modelos de Predicción y Clasificación

Notebook de **evidencias**. Usa las mismas clases y la misma configuración (`src/config.py`)
que `main.py`, el script que ejecuta GitHub Actions.

```
Hugging Face → DataFrame → preprocesamiento
                         → regresión lineal + evaluación
                         → clasificación + evaluación
                         → resumen automático .txt
```

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.pipeline import (load_raw_data, build_preprocessor, regression_subset,
                          build_regression, build_classifier, run_experiment)
from src.preprocessing import split_data
from src.classification_model import TextLogisticClassifier
from src.interpretation import (interpret_regression, interpret_classification,
                                describe_class_balance)
from src.report_writer import ModelSummaryWriter

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Semilla:", config.RANDOM_STATE, "| test_size:", config.TEST_SIZE)

Semilla: 42 | test_size: 0.3


## 0. Construcción automática del DataFrame

| Campo | Valor |
|---|---|
| Identificador | `manuelgscom/dale-bash-test-v2` |
| Enlace | https://huggingface.co/datasets/manuelgscom/dale-bash-test-v2 |
| Split | `test` (único publicado; configuración `default`) |
| Contenido | Peticiones en lenguaje natural (es/en) a un asistente de terminal Linux |

Columnas:
- `request`: la petición.
- `lang`: el idioma.
- `category`: el tema.
- `context`: un diccionario con distro, gestor de paquetes y herramientas instaladas.
- `expected_type`: lo que debe hacer el asistente (`command`, `refuse` o `unsure`).
- `reference_commands`: los comandos bash correctos.
- `notes`: comentario para evaluadores.
- `id`: identificador.

- **Clasificación:** objetivo categórico → `expected_type`. Es una etiqueta original del dataset.
- **Regresión:** el dataset no trae una variable numérica continua, así que se **construye**
  `command_length`, la longitud en caracteres del comando de referencia, como medida de la
  complejidad de la petición. Solo se usa como objetivo; `reference_commands` nunca entra como predictor.

In [2]:
df_raw = load_raw_data()

print("Dataset:", config.DATASET_ID, "| split:", config.DATASET_SPLIT)
print("Tamaño (filas, columnas):", df_raw.shape)
print("Columnas:", list(df_raw.columns))
df_raw.dtypes

Dataset: manuelgscom/dale-bash-test-v2 | split: test
Tamaño (filas, columnas): (471, 8)
Columnas: ['id', 'lang', 'category', 'context', 'request', 'expected_type', 'reference_commands', 'notes']


id                       str
lang                     str
category                 str
context               object
request                  str
expected_type            str
reference_commands    object
notes                    str
dtype: object

In [3]:
df_raw.head()

,id,lang,category,context,request,expected_type,reference_commands,notes
0,dev3-ficheros-es-001,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-22.04', 'pm': 'apt', 'tools': ['batcat', 'fdfind', '...","Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos los ocultos...",command,[tree -a -d /srv/musica],"[contexto] Debe usar tree para mostrar recursivamente solo directorios, incluidos los ..."
1,dev3-ficheros-es-002,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-24.04', 'pm': 'apt', 'tools': []}","Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/imagenes, co...",command,[mkdir -p -- '/home/ana/Proyecto Sur/documentos' '/home/ana/Proyecto Sur/imagenes'],"Debe crear ambos directorios y los padres necesarios, respetando el espacio del nombre."
2,dev3-ficheros-es-003,es,ficheros,"{'os': 'linux', 'distro': 'debian-12', 'pm': 'apt', 'tools': []}",Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/contrato.odt ...,command,[cp -p -- /home/ines/borradores/contrato.odt /mnt/respaldo/contrato.odt],[contexto][ausente] Falta rsync; cp -p permite copiar el fichero conservando los permi...
3,dev3-ficheros-es-004,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-22.04', 'pm': 'apt', 'tools': ['sudo']}",Borra definitivamente todos los ficheros regulares cuyo nombre termine en .bak dentro ...,command,[sudo find /srv/contabilidad -type f -name '*.bak' -delete],"[peligroso] Debe borrar recursivamente solo ficheros regulares terminados en .bak, sin..."
4,dev3-ficheros-es-005,es,ficheros,"{'os': 'linux', 'distro': 'ubuntu-24.04', 'pm': 'apt', 'tools': ['curl', 'git', 'jq', ...","Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificación más reci...",command,[ls -lt -- /srv/entregas],Debe mostrar el listado detallado ordenado por fecha de modificación descendente sin i...


In [4]:
print("Ejemplo de context:", df_raw["context"].iloc[0])
print("Ejemplo de reference_commands:", list(df_raw["reference_commands"].iloc[0]))

Ejemplo de context: {'os': 'linux', 'distro': 'ubuntu-22.04', 'pm': 'apt', 'tools': array(['batcat', 'fdfind', 'rg', 'tree'], dtype=object)}
Ejemplo de reference_commands: ['tree -a -d /srv/musica']


In [5]:
for column in ["lang", "expected_type"]:
    print(df_raw[column].value_counts(), "\n")

lang
es    257
en    214
Name: count, dtype: int64 

expected_type
command    339
refuse     101
unsure      31
Name: count, dtype: int64 



### Detección de fuga de información
La tabla cruzada muestra que `category` **revela la respuesta** del clasificador:
`fuera_de_ambito` siempre es `refuse` y `ambigua` siempre es `unsure`. Por eso `category`
**no** se usa como predictor de `expected_type`.

In [6]:
pd.crosstab(df_raw["category"], df_raw["expected_type"])

expected_type,command,refuse,unsure
category,,,
ambigua,0,0,31
compresion,23,0,0
discos,22,0,0
docker,19,0,0
ficheros,42,0,0
fuera_de_ambito,0,101,0
git,28,0,0
misc,25,0,0
paquetes,28,0,0


## 1. Preprocesamiento

### 1.1 Estado inicial

In [7]:
print("Nulos por columna:")
print(df_raw.isnull().sum())
print()
print("Peticiones duplicadas:", int(df_raw["request"].duplicated().sum()))
print("Valores distintos de os:", pd.json_normalize(df_raw["context"].tolist())["os"].unique())
print("Tipos de las columnas con listas:",
      type(df_raw["reference_commands"].iloc[0]).__name__, "/",
      type(df_raw["context"].iloc[0]).__name__)

Nulos por columna:
id                    0
lang                  0
category              0
context               0
request               0
expected_type         0
reference_commands    0
notes                 0
dtype: int64

Peticiones duplicadas: 0
Valores distintos de os: <ArrowStringArray>
['linux']
Length: 1, dtype: str
Tipos de las columnas con listas: ndarray / dict


### 1.2 Limpieza y enriquecimiento (`DataPreprocessor.clean`)

Estas operaciones **no aprenden nada de los datos**, así que se aplican antes de dividir:

1. **Expandir `context`** en columnas planas (`os`, `distro`, `pm`, `tools`).
2. **Eliminar columnas**:
   - `id`: es un identificador.
   - `os`: es constante.
   - `notes`: anticipa la respuesta, lo que sería fuga de información.
3. **Nulos y duplicados** en la petición: no hay, pero se controla.
4. **Corregir tipos**: `lang`, `category`, `distro`, `pm` y `expected_type` pasan a `category`.
5. **Enriquecer**:
   - `request_length`: caracteres de la petición.
   - `n_paths`: rutas como `/srv/fotos` mencionadas en la petición.
   - `n_tools`: herramientas instaladas.
   - `command_length`: objetivo de la regresión.
6. **Eliminar las listas** `tools` y `reference_commands` una vez resumidas.

In [8]:
preprocessor = build_preprocessor()
df_clean = preprocessor.clean(df_raw)
preprocessor.steps_as_dataframe()

,operation,reason,effect
0,Expandir columna anidada,'context' es un diccionario; los modelos necesitan columnas planas.,"context -> ['os', 'distro', 'pm', 'tools']"
1,Eliminar columnas,id: Identificador único: no describe la petición. os: Vale 'linux' en todas las filas:...,"Columnas eliminadas: ['id', 'os', 'notes']."
2,Nulos y duplicados,Una petición vacía no se puede modelar y una repetida en train y test infla las métricas.,Nulos: 0; peticiones duplicadas: 0; filas: 471 -> 471.
3,Corregir tipos,Estas columnas toman pocos valores fijos: son categóricas.,"['lang', 'category', 'distro', 'pm', 'expected_type'] -> tipo 'category'."
4,Enriquecer con variables numéricas,El texto y las listas no son numéricos; se resumen en variables que los modelos pueden...,"request_length (caracteres de la petición), n_paths (rutas mencionadas), n_tools (herr..."
5,Eliminar listas ya resumidas,Las listas originales no son utilizables por los modelos.,"Columnas eliminadas: ['tools', 'reference_commands']; quedan 10."


In [9]:
comparison = pd.DataFrame({
    "antes": [df_raw.shape[0], df_raw.shape[1], int(df_raw.isnull().sum().sum()),
              "context, reference_commands", "ninguna", "texto (str)"],
    "después": [df_clean.shape[0], df_clean.shape[1], int(df_clean.drop(columns="command_length").isnull().sum().sum()),
                "ninguna", "request_length, n_paths, n_tools, command_length", "category"],
}, index=["filas", "columnas", "nulos", "columnas anidadas/listas", "variables numéricas", "tipo de lang/category/..."])
comparison

,antes,después
filas,471,471
columnas,8,10
nulos,0,0
columnas anidadas/listas,"context, reference_commands",ninguna
variables numéricas,ninguna,"request_length, n_paths, n_tools, command_length"
tipo de lang/category/...,texto (str),category


In [10]:
df_clean.head()

,lang,category,request,expected_type,distro,pm,request_length,n_paths,n_tools,command_length
0,es,ficheros,"Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos los ocultos...",command,ubuntu-22.04,apt,114,1,4,22.0
1,es,ficheros,"Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/imagenes, co...",command,ubuntu-24.04,apt,128,2,0,81.0
2,es,ficheros,Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/contrato.odt ...,command,debian-12,apt,174,2,0,70.0
3,es,ficheros,Borra definitivamente todos los ficheros regulares cuyo nombre termine en .bak dentro ...,command,ubuntu-22.04,apt,151,1,1,57.0
4,es,ficheros,"Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificación más reci...",command,ubuntu-24.04,apt,144,1,4,23.0


In [11]:
df_clean.dtypes

lang              category
category          category
request                str
expected_type     category
distro            category
pm                category
request_length       int64
n_paths              int64
n_tools              int64
command_length     float64
dtype: object

In [12]:
df_clean.groupby("expected_type", observed=True)[["request_length", "n_paths", "n_tools", "command_length"]].mean().round(2)

,request_length,n_paths,n_tools,command_length
expected_type,,,,
command,109.95,0.33,2.59,38.42
refuse,104.00,0.00,2.50,NaN
unsure,75.00,0.45,1.97,NaN


`command_length` está vacía (NaN) en las peticiones `refuse` y `unsure`, porque no tienen comando.
Por eso la regresión usa solo las filas `command`.

In [13]:
# Ejemplos de la nueva variable n_paths
df_clean.loc[df_clean["n_paths"] > 0, ["request", "n_paths"]].head()

,request,n_paths
0,"Enséñame en forma de árbol todos los directorios de /srv/musica, incluidos los ocultos...",1
1,"Necesito crear /home/ana/Proyecto Sur/documentos y /home/ana/Proyecto Sur/imagenes, co...",2
2,Usa rsync para copiar /home/ines/borradores/contrato.odt a /mnt/respaldo/contrato.odt ...,2
3,Borra definitivamente todos los ficheros regulares cuyo nombre termine en .bak dentro ...,1
4,"Lista las entradas de /srv/entregas, sin incluir las ocultas, de modificación más reci...",1


### 1.3 Transformaciones que aprenden de los datos (sin fuga)

El **escalado** (`StandardScaler`), la **codificación** (`OneHotEncoder`) y el **vocabulario TF-IDF**
aprenden parámetros. Por eso van dentro de un `Pipeline` que se ajusta **solo con `X_train`**; la
evidencia está en cada modelo.

En ambos modelos la división es 70 % / 30 %, `random_state=42` y estratificada:
- La regresión estratifica por `category`.
- La clasificación estratifica por `expected_type`.

## 2. Modelo de regresión lineal

**Pregunta:** ¿se puede estimar **qué tan largo (complejo) será el comando** necesario para resolver
una petición a partir de rasgos de la propia petición?

- **Datos:** solo peticiones `command` (las únicas con comando).
- **Objetivo:** `command_length`, en caracteres.
- **Predictoras:**
  - `request_length`: una petición más detallada suele pedir más opciones.
  - `n_paths`: cada ruta mencionada aparece en el comando.
  - `n_tools`: herramientas disponibles.
  - `lang`.
  - `category`: el tema; por ejemplo, git y paquetes tienen comandos típicos de distinta longitud.
- **Algoritmo:** `LinearRegression` con **parámetros predeterminados**.
- **Pipeline:**
  - `StandardScaler` en las numéricas.
  - `OneHotEncoder(drop="first")` en `lang` y `category`; la referencia es la primera categoría.

In [14]:
df_regression = regression_subset(df_clean)
print("Filas para regresión:", len(df_regression))
df_regression[config.REGRESSION_TARGET].describe().round(2)

Filas para regresión: 339


count    339.00
mean      38.42
std       22.91
min        3.00
25%       23.00
50%       33.00
75%       48.00
max      188.00
Name: command_length, dtype: float64

In [15]:
# Relación lineal de cada predictora numérica con el objetivo
df_regression[config.REGRESSION_NUMERIC_FEATURES + [config.REGRESSION_TARGET]].corr().round(3)[[config.REGRESSION_TARGET]]

,command_length
request_length,0.476
n_paths,0.293
n_tools,0.088
command_length,1.000


In [16]:
regression = build_regression()
Xr_train, Xr_test, yr_train, yr_test = split_data(
    df_regression, regression.features, config.REGRESSION_TARGET,
    config.TEST_SIZE, config.RANDOM_STATE, stratify_column=config.REGRESSION_STRATIFY)
regression.fit(Xr_train, yr_train)

print("Entrenamiento:", Xr_train.shape, "| Prueba:", Xr_test.shape)
print("Parámetros de LinearRegression:", regression.params())

Entrenamiento: (237, 5) | Prueba: (102, 5)
Parámetros de LinearRegression: {'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}


**Evidencia de que el escalado se ajustó solo con entrenamiento:** las medias del `StandardScaler`
coinciden con las de `X_train`. Prueba queda cerca de 0, pero no exactamente.

In [17]:
preprocess = regression.pipeline.named_steps["preprocess"]
scaler = preprocess.named_transformers_["num"]
num = config.REGRESSION_NUMERIC_FEATURES
print("¿Medias del scaler == medias de X_train?", np.allclose(scaler.mean_, Xr_train[num].mean()))

names = preprocess.get_feature_names_out()
to_dense = lambda m: m.toarray() if hasattr(m, "toarray") else m
train_t = pd.DataFrame(to_dense(preprocess.transform(Xr_train)), columns=names)
test_t = pd.DataFrame(to_dense(preprocess.transform(Xr_test)), columns=names)
pd.DataFrame({
    "media train antes": Xr_train[num].mean(),
    "media train después": train_t[num].mean(),
    "desv. train después": train_t[num].std(ddof=0),
    "media test después": test_t[num].mean(),
}).round(3)

¿Medias del scaler == medias de X_train? True


,media train antes,media train después,desv. train después,media test después
request_length,109.675,0.0,1.0,0.028
n_paths,0.354,0.0,1.0,-0.134
n_tools,2.684,-0.0,1.0,-0.165


In [18]:
print("Variables tras one-hot:", list(names))
print("Referencias (drop='first'):", regression.reference_categories())

Variables tras one-hot: ['request_length', 'n_paths', 'n_tools', 'lang_es', 'category_discos', 'category_docker', 'category_ficheros', 'category_git', 'category_misc', 'category_paquetes', 'category_permisos', 'category_procesos', 'category_red', 'category_sistema', 'category_texto']
Referencias (drop='first'): {'lang': 'en', 'category': 'compresion'}


### 2.1 Evaluación en prueba

In [19]:
reg_metrics = regression.evaluate(Xr_test, yr_test)
reg_cv = regression.cross_validate(Xr_train, yr_train, config.CV_FOLDS)
pd.Series({
    "MAE (caracteres)": reg_metrics["mae"],
    "RMSE (caracteres)": reg_metrics["rmse"],
    "R²": reg_metrics["r2"],
    "MAE modelo base (media)": reg_metrics["baseline_mae"],
    f"R² CV {config.CV_FOLDS} pliegues (media)": reg_cv["r2_mean"],
    f"R² CV {config.CV_FOLDS} pliegues (desv.)": reg_cv["r2_std"],
}).round(4)

MAE (caracteres)            14.7811
RMSE (caracteres)           22.4037
R²                           0.2404
MAE modelo base (media)     17.2235
R² CV 5 pliegues (media)     0.2019
R² CV 5 pliegues (desv.)     0.0727
dtype: float64

In [20]:
predictions = df_regression.loc[Xr_test.index, ["request", "category"]].copy()
predictions["real"] = yr_test
predictions["predicho"] = regression.predict(Xr_test).round(1)
predictions["error_abs"] = (predictions["real"] - predictions["predicho"]).abs().round(1)
predictions.sort_values("error_abs", ascending=False).head(8)

,request,category,real,predicho,error_abs
66,"Convert the UTF-8 tab-separated inventory.tsv to CSV on stdout, preserving the header ...",texto,188.0,53.5,134.5
207,"Give me one JSON array listing block devices whose type is disk, with only path and si...",discos,119.0,53.9,65.1
253,Enséñame la versión de HEAD de docs/instalacion.md con resaltado de sintaxis Markdown ...,git,92.0,39.8,52.2
25,Rename /home/amy/Invoices/April draft.pdf to April final.pdf in the same folder.,ficheros,79.0,33.0,46.0
79,"Print the unique lines that occur in approved.txt but not in revoked.txt, sorted in C ...",texto,82.0,39.5,42.5
135,"Show all listening TCP sockets with numeric addresses and ports, including the owning ...",red,13.0,53.7,40.7
17,"Con fd, localiza los directorios llamados exactamente node_modules bajo /home/sofia/pr...",ficheros,88.0,51.2,36.8
75,Convert customers-latin1.csv from ISO-8859-1 to UTF-8 and save it as customers-utf8.cs...,texto,70.0,34.4,35.6


- **MAE:** error promedio en caracteres.
- **RMSE:** penaliza los errores grandes. Que sea bastante mayor que el MAE indica algunos comandos
  muy largos que el modelo no anticipa.
- **R²:** proporción de la variabilidad explicada.
- **Modelo base:** predice siempre la media; sirve de referencia.

### 2.2 Coeficientes e intercepto

In [21]:
print(regression.coefficients().round(3).to_string(index=False))
print()
for name, value in regression.intercepts().items():
    print(f"{name}: {value:.3f}")

         variable  coef_estandarizado  coef_por_unidad
   request_length               8.433            0.264
          n_paths               4.766            7.994
          n_tools               2.003            1.096
          lang_es              -2.514           -2.514
  category_discos               1.847            1.847
  category_docker               5.264            5.264
category_ficheros              -1.510           -1.510
     category_git               2.747            2.747
    category_misc               4.542            4.542
category_paquetes             -11.926          -11.926
category_permisos              -4.185           -4.185
category_procesos              -4.531           -4.531
     category_red               3.373            3.373
 category_sistema              -0.020           -0.020
   category_texto               0.005            0.005

intercepto_estandarizado: 40.121
intercepto_unidades_originales: 5.386


- `coef_estandarizado`: cambio en caracteres del comando por cada **desviación estándar** de la
  variable, en la escala del Pipeline.
- `coef_por_unidad`: el mismo efecto **por cada carácter, ruta o herramienta** adicional.
- `lang_es` y `category_*`: diferencia frente a la categoría de referencia con las mismas
  condiciones.
- El intercepto estandarizado es la predicción para las categorías de referencia con valores
  numéricos promedio.

In [22]:
for line in interpret_regression(regression, reg_metrics, reg_cv, config.CV_FOLDS,
                                 config.REGRESSION_UNIT, config.FEATURE_UNITS):
    print("-", line)

- R² = 0.240: el modelo explica el 24.0% de la variabilidad de command_length en los datos de prueba; el resto depende de factores que las variables no capturan.
- MAE = 14.78 caracteres: en promedio la predicción se aleja 14.8 caracteres del valor real (37.5% del valor medio, 39.4 caracteres).
- RMSE = 22.40 caracteres (1.52 veces el MAE): hay algunos errores grandes (comandos mucho más largos de lo esperado) que pesan más que el resto.
- Referencia: predecir siempre la media daría MAE = 17.22 caracteres; la regresión reduce el error en 14.2%.
- Validación cruzada (5 pliegues sobre entrenamiento): R² = 0.202 ± 0.073; el desempeño varía bastante según la partición.
- request_length: +0.264 caracteres de command_length por cada carácter de la petición adicional (+8.43 por desviación estándar), con las demás variables constantes.
- n_paths: +7.994 caracteres de command_length por cada ruta mencionada adicional (+4.77 por desviación estándar), con las demás variables constantes.
- n_tools

**Limitaciones:**
- El R² es modesto: la longitud de un comando depende sobre todo de su sintaxis concreta
  (opciones, tuberías, comillas), que la petición no revela.
- El objetivo es un conteo construido y con cola larga.
- Las categorías tienen pocos ejemplos cada una, así que sus coeficientes son inestables (ver la
  desviación de la validación cruzada).
- El modelo es lineal y no captura interacciones.

## 3. Modelo de clasificación

**Qué se clasifica:** el `expected_type` de cada petición, es decir, si el asistente debe:
- `command`: responder con un comando;
- `refuse`: rechazarla porque está fuera de ámbito, como una charla o una pregunta ajena a la terminal;
- `unsure`: pedir aclaración porque es ambigua.

**Predictoras:**
- El **texto de la petición** (`request`) vectorizado con TF-IDF.
- `request_length` y `n_paths`.

Se excluyen por **fuga de información**:
- `category`: equivale a la clase.
- `reference_commands`: está vacía cuando no hay comando.
- `command_length`: se deriva de `reference_commands`.

**Algoritmo: TF-IDF + regresión logística.**
- TF-IDF convierte cada petición en pesos de palabras y bigramas, con `ngram_range=(1, 2)`,
  `min_df=2` y `sublinear_tf=True`.
- La regresión logística es lineal, rápida, funciona bien con textos cortos y pocos datos, y es
  interpretable: cada término tiene un peso por clase.
- **Hiperparámetros:**
  - `class_weight="balanced"`: compensa el desbalance de clases.
  - `max_iter=1000`.
  - `random_state=42`.
  - El resto son predeterminados: `C=1.0`, `solver="lbfgs"`, L2.

In [23]:
distribution = df_clean["expected_type"].value_counts().sort_index()
print(describe_class_balance(distribution))
distribution

Distribución de clases: command=339 (72%), refuse=101 (21%), unsure=31 (7%). Están desbalanceadas: la exactitud premia a la clase mayoritaria, así que se priorizan recall y F1 macro (cada clase pesa igual) y se usa class_weight='balanced'.


expected_type
command    339
refuse     101
unsure      31
Name: count, dtype: int64

In [24]:
classifier = build_classifier()
Xc_train, Xc_test, yc_train, yc_test = split_data(
    df_clean, classifier.features, config.CLASSIFICATION_TARGET,
    config.TEST_SIZE, config.RANDOM_STATE, stratify_column=config.CLASSIFICATION_TARGET)
classifier.fit(Xc_train, yc_train)

print("Entrenamiento:", Xc_train.shape, "| Prueba:", Xc_test.shape)
print("Clases en prueba:", yc_test.value_counts().to_dict())
print("Hiperparámetros:", classifier.params())

Entrenamiento: (329, 3) | Prueba: (142, 3)


Clases en prueba: {'command': 102, 'refuse': 31, 'unsure': 9}
Hiperparámetros: {'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


**Evidencia de que TF-IDF no usa información de prueba:** el vocabulario se aprende solo con
entrenamiento. Las palabras que aparecen únicamente en prueba se **ignoran** al transformar.

In [25]:
vocabulary = classifier.pipeline.named_steps["preprocess"].named_transformers_["text"].vocabulary_
test_words = set(" ".join(Xc_test["request"]).lower().split())
print("Términos en el vocabulario (aprendido con train):", len(vocabulary))
print("Palabras de prueba que no están en el vocabulario:", len(test_words - set(vocabulary)))

Términos en el vocabulario (aprendido con train): 1132
Palabras de prueba que no están en el vocabulario: 912


### 3.1 Evaluación: matriz de confusión y métricas

Como las clases están **desbalanceadas** (72 % `command`), la exactitud por sí sola engaña. Un
modelo que siempre dijera `command` acertaría el 72 %. Por eso se priorizan el **recall** y el
**F1 macro**, que dan el mismo peso a cada clase.

In [26]:
clf_metrics = classifier.evaluate(Xc_test, yc_test)
clf_cv = classifier.cross_validate(Xc_train, yc_train, config.CV_FOLDS, config.RANDOM_STATE)
clf_metrics["confusion_matrix"]

,pred_command,pred_refuse,pred_unsure
real_command,87,7,8
real_refuse,3,24,4
real_unsure,2,1,6


In [27]:
print(clf_metrics["report_text"])
print(f"F1 macro CV ({config.CV_FOLDS} pliegues): {clf_cv['f1_macro_mean']:.3f} ± {clf_cv['f1_macro_std']:.3f}")

              precision    recall  f1-score   support

     command      0.946     0.853     0.897       102
      refuse      0.750     0.774     0.762        31
      unsure      0.333     0.667     0.444         9

    accuracy                          0.824       142
   macro avg      0.676     0.765     0.701       142
weighted avg      0.864     0.824     0.839       142

F1 macro CV (5 pliegues): 0.628 ± 0.096


**Justificación de `class_weight="balanced"`:** el mismo modelo sin balanceo, con la misma división:

In [28]:
unbalanced = TextLogisticClassifier(
    config.CLASSIFICATION_TEXT_FEATURE, config.CLASSIFICATION_NUMERIC_FEATURES,
    config.CLASSIFICATION_TARGET, ngram_range=config.TFIDF_NGRAM_RANGE,
    min_df=config.TFIDF_MIN_DF, class_weight=None,
    max_iter=config.LOGISTIC_MAX_ITER, random_state=config.RANDOM_STATE).fit(Xc_train, yc_train)
unbalanced_metrics = unbalanced.evaluate(Xc_test, yc_test)
print("F1 macro sin balanceo:", round(unbalanced_metrics["f1_macro"], 3),
      "| con balanceo:", round(clf_metrics["f1_macro"], 3))
unbalanced_metrics["confusion_matrix"]

F1 macro sin balanceo: 0.423 | con balanceo: 0.701


,pred_command,pred_refuse,pred_unsure
real_command,102,0,0
real_refuse,23,8,0
real_unsure,8,1,0


In [29]:
# Peticiones de prueba mal clasificadas
errors = df_clean.loc[Xc_test.index, ["request"]].copy()
errors["real"] = yc_test.astype(str)
errors["predicho"] = classifier.predict(Xc_test)
errors[errors["real"] != errors["predicho"]].head(12)

,request,real,predicho
149,"Necesito pip para Python 3, instálalo desde los repositorios de Fedora.",command,refuse
443,Abre en el cortafuegos el puerto que necesita nuestro servicio de métricas.,unsure,command
431,Turn this premise into a 900-word short story: a lighthouse keeper receives letters da...,refuse,command
346,"A ver si sacas esta adivinanza: tengo ciudades sin casas, montañas sin árboles y agua ...",refuse,command
187,Power off this computer now; I'm finished using it.,command,unsure
421,Why does the Moon look bigger near the horizon?,refuse,unsure
205,Necesito consultar la tabla de particiones de /dev/nvme0n1 sin modificarla.,command,unsure
150,¿Qué rutas de archivos y directorios registra el paquete tzdata que tengo instalado?,command,refuse
398,Tell me a quick joke about a forgetful librarian.,refuse,unsure
400,How has some ancient Roman concrete survived for so long?,refuse,unsure


In [30]:
pd.DataFrame(classifier.top_features(8)).rename_axis("posición").T

posición,0,1,2,3,4,5,6,7
command,sin,tiene,ya,request_length,txt,print,n_paths,ahora
refuse,qué,un,my,is,cómo,sobre,me,pulpos
unsure,que,to the,n_paths,to,que te,para el,ponle,al


In [31]:
for line in interpret_classification(classifier, clf_metrics, clf_cv, config.CV_FOLDS):
    print("-", line)

- Exactitud = 0.824: 117 de 142 peticiones de prueba se clasificaron correctamente.
- F1 macro = 0.701 (precisión macro = 0.676, recall macro = 0.765).
- Clase command: recall 0.853 (detecta 85% de los casos reales), precisión 0.946, F1 0.897, 102 casos en prueba.
- Clase refuse: recall 0.774 (detecta 77% de los casos reales), precisión 0.750, F1 0.762, 31 casos en prueba.
- Clase unsure: recall 0.667 (detecta 67% de los casos reales), precisión 0.333, F1 0.444, 9 casos en prueba.
- Mejor reconocida: command; más difícil: unsure.
- Errores (de más a menos frecuente): 8 command predicha(s) como unsure; 7 command predicha(s) como refuse; 4 refuse predicha(s) como unsure; 3 refuse predicha(s) como command; 2 unsure predicha(s) como command; 1 unsure predicha(s) como refuse.
- Validación cruzada estratificada (5 pliegues sobre entrenamiento): F1 macro = 0.628 ± 0.096.
- Términos que más empujan hacia command: sin, tiene, ya, request_length, txt.
- Términos que más empujan hacia refuse: qué

**Interpretación:**
- `command` es la clase mejor reconocida: es la mayoritaria, y la empujan las peticiones más largas,
  las que mencionan rutas (`n_paths`) o extensiones (`txt`) y las que ponen restricciones
  ("sin", "ya", "tiene").
- Hacia `refuse` empujan palabras de pregunta o charla ("qué", "cómo", "is", "sobre").
- `refuse` se identifica bastante bien porque son preguntas de charla o temas ajenos a la terminal.
- `unsure` es la más difícil: tiene pocos ejemplos, y una petición ambigua usa las mismas palabras
  que una válida. Sin el balanceo el modelo nunca la predice.
- Los errores más frecuentes son peticiones válidas marcadas como `unsure` o `refuse`, es decir,
  el modelo se vuelve conservador.

**Limitaciones:**
- Hay pocos datos y son bilingües.
- TF-IDF solo ve qué palabras aparecen, no su significado.
- Con 9 casos `unsure` en prueba, cada acierto o error de esa clase mueve mucho sus métricas.

## 4. Resumen automático `.txt`

Se ejecuta `run_experiment()`, el mismo flujo que corre `main.py` en GitHub Actions. Se comprueba
que sus métricas coinciden con las de este notebook y se genera el resumen.

In [32]:
results = run_experiment()

assert np.isclose(results.regression_metrics["r2"], reg_metrics["r2"])
assert np.isclose(results.regression_metrics["mae"], reg_metrics["mae"])
assert np.isclose(results.classification_metrics["f1_macro"], clf_metrics["f1_macro"])
assert results.classification_metrics["confusion_matrix"].equals(clf_metrics["confusion_matrix"])
print("Las métricas del flujo automático coinciden con las del notebook.")

OUTPUT_PATH = PROJECT_ROOT / "output" / config.SUMMARY_FILENAME
saved_path = ModelSummaryWriter(results).save(OUTPUT_PATH)
print("Resumen guardado en:", saved_path)
print("¿Existe?", saved_path.exists(), "| tamaño:", saved_path.stat().st_size, "bytes")

Las métricas del flujo automático coinciden con las del notebook.
Resumen guardado en: C:\Proyectos\MachineLearningIU\S30_Unidad1_EA2_Modelos_de_Prediccion_y_Clasificacion\output\resumen_modelos.txt
¿Existe? True | tamaño: 12539 bytes


In [33]:
print(saved_path.read_text(encoding="utf-8"))

S30 - EA2 UNIDAD 1 - MODELOS DE PREDICCIÓN Y CLASIFICACIÓN
Resumen generado automáticamente por código
Fecha de ejecución: 2026-10-28 22:04:57 (UTC-0500)
Entorno: Python 3.12.10 | pandas 3.1.0 | scikit-learn 1.9.1 | datasets 5.1.0

1. DATASET
Identificador: manuelgscom/dale-bash-test-v2
Enlace:        https://huggingface.co/datasets/manuelgscom/dale-bash-test-v2
Split:         test (único split publicado)
Contenido:     peticiones en lenguaje natural (es/en) a un asistente de terminal Linux, con el tipo de respuesta esperado y comandos bash de referencia.
Obtención:     automática con datasets.load_dataset() -> pandas
Tamaño original: 471 filas x 8 columnas
Columnas originales: ['id', 'lang', 'category', 'context', 'request', 'expected_type', 'reference_commands', 'notes']
Tamaño limpio:   471 filas x 10 columnas
Columnas limpias:    ['lang', 'category', 'request', 'expected_type', 'distro', 'pm', 'request_length', 'n_paths', 'n_tools', 'command_length']

2. PREPROCESAMIENTO
Operacione

## 5. Ejecución automática en GitHub Actions

El workflow `.github/workflows/s30_modelos_ml.yml` (en la raíz del repositorio) se activa con cada
**push** a `main` que modifique esta carpeta. Hace lo siguiente:

1. Instala Python 3.12 y `requirements.txt`.
2. Ejecuta `python main.py`: descarga el dataset, lo preprocesa, entrena y evalúa los dos modelos
   y genera `output/resumen_modelos.txt`.
3. Publica el resumen como **artefacto descargable** `resumen-modelos`.